In [1]:
!pip install evaluate
!pip install --upgrade transformers

In [2]:
import torch
import numpy as np
import gradio as gr

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
import evaluate

In [3]:
# AG News datasetni yuklaymiz
dataset = load_dataset("ag_news")

# Train datasetdan 2000 ta sample olamiz
train_dataset = dataset["train"].shuffle(seed=42).select(range(2000))

# Test datasetdan 400 ta sample olamiz
test_dataset = dataset["test"].shuffle(seed=42).select(range(400))

print("Train size:", len(train_dataset))
print("Test size:", len(test_dataset))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train size: 2000
Test size: 400


In [4]:
# Klasslar nomi
label_names = ["World", "Sports", "Business", "Sci/Tech"]

# ID -> label
id2label = {i: label for i, label in enumerate(label_names)}

# label -> ID
label2id = {label: i for i, label in enumerate(label_names)}

In [5]:
# BERT tokenizer yuklaymiz
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Tokenlash funksiyasi
def tokenize_function(examples):
    return tokenizer(
        examples["text"],          # matn
        truncation=True,           # uzun matnni kesish
        padding="max_length",      # padding qo'shish
        max_length=256             # maksimal uzunlik
    )

# Train datasetni tokenlash
tokenized_train = train_dataset.map(tokenize_function, batched=True)

# Test datasetni tokenlash
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

In [6]:
# Text ustunini olib tashlaymiz (modelga kerak emas)
tokenized_train = tokenized_train.remove_columns(["text"])
tokenized_test = tokenized_test.remove_columns(["text"])

# Torch formatga o'tkazamiz
tokenized_train.set_format("torch")
tokenized_test.set_format("torch")

In [7]:
# BERT modelni yuklaymiz (4 ta label uchun)
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [8]:
# Accuracy metric yuklaymiz
accuracy = evaluate.load("accuracy")

# Metric hisoblash funksiyasi
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [10]:
training_args = TrainingArguments(
    output_dir="./results",              # model natijalari saqlanadi
    per_device_train_batch_size=8,       # train batch size
    per_device_eval_batch_size=8,        # eval batch size
    num_train_epochs=1,                  # 1 epoch
    eval_strategy="epoch",               # eski evaluation_strategy o'rniga shu
    save_strategy="epoch",               # har epochdan keyin saqlash
    logging_steps=50,                    # har 50 stepda log
    report_to="none"                     # tashqi loggingni o'chirish
)

In [12]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

In [14]:
# Training boshlash
trainer.train()



Epoch,Training Loss,Validation Loss,Accuracy
1,0.274189,0.473142,0.887500


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=250, training_loss=0.27046907806396486, metrics={'train_runtime': 117.1367, 'train_samples_per_second': 17.074, 'train_steps_per_second': 2.134, 'total_flos': 263115780096000.0, 'train_loss': 0.27046907806396486, 'epoch': 1.0})

In [15]:
# Device tanlash (GPU bo'lsa ishlatadi)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model.to(device)
model.eval()

# Bashorat funksiyasi
def predict_news_category(text):
    # Tokenlash
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=256
    )

    # GPU/CPU ga o'tkazish
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # Prediction
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        predicted_class_id = torch.argmax(logits, dim=1).item()

    return id2label[predicted_class_id]

In [16]:
# Model weights saqlash
torch.save(model.state_dict(), "news_model.pth")

print("Model saved as news_model.pth")

Model saved as news_model.pth


In [17]:
# Gradio interfeys
interface = gr.Interface(
    fn=predict_news_category,
    inputs=gr.Textbox(lines=6, placeholder="Yangilik matnini kiriting..."),
    outputs=gr.Textbox(label="Natija"),
    title="Yangilik turini aniqlash",
    description="Bu yerda yangilik matnini kiritib, uning turini bilib oling"
)

# Ishga tushirish
interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c2cacfca79e6cf829d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
